# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library and Croissant schema definitions.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets with their @id and fields
record_sets = []
print("Available Record Sets:\n")
for rs in metadata.record_sets:
    print(f" - RecordSet: {rs['@id']}  | name: {rs['name']}")
    field_ids = [field['@id'] for field in rs.get('fields', [])]
    print(f"     Fields (by @id): {field_ids}")
    record_sets.append(rs['@id'])
if not record_sets:
    print("No record sets were defined. The dataset schema may use different conventions; proceeding to enumerate columns in file objects if available.")
    if hasattr(metadata, 'data_files'):
        for fo in metadata.data_files:
            print(f" - DataFileObject: {fo['@id']}")
            if 'columns' in fo:
                col_ids = [col['@id'] for col in fo['columns']]
                print(f"     Columns (by @id): {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# The FAIR^2 dataset schema may use only a single main record set for tabular data.
# We'll load all available record sets (if provided). Otherwise, detect main data files.
from pprint import pprint

# Use the record_sets list from the previous cell
dataframes = {}

if record_sets:
    for rs_id in record_sets:
        print(f"\nLoading records for RecordSet: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Fields/Columns: {list(df.columns)}")
        print(df.head())
else:
    # If no record sets, try to read from the automatically detected primary tabular data file
    print("No record sets detected. Attempting to load main data table...")
    try:
        # dataset.records() without arguments yields primary data
        records = list(dataset.records())
        main_df = pd.DataFrame(records)
        dataframes['main'] = main_df
        print(f"Columns: {main_df.columns.tolist()}")
        print(main_df.head())
    except Exception as e:
        print("Error loading data records:", e)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. Here we'll demonstrate with proteins' abundance or a similar numeric field.

In [ ]:
# Select the DataFrame and a numeric column for analysis
if dataframes:
    # Pick the first available DataFrame
    df_key = next(iter(dataframes))
    df = dataframes[df_key]
    print(f"Working with DataFrame: {df_key}")

    # Show all numeric columns
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    print("Numeric columns:", numeric_cols)
    if not numeric_cols:
        # Attempt to convert columns with numeric values
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include='number').columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]  # Choose the first numeric field
        threshold = df[numeric_field].quantile(0.7)
        # For demonstration, filter values above this quantile
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
        display_cols = [numeric_field]
        print(filtered_df[display_cols].head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print("\nNormalized sample:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a likely group field (categorical)
        group_fields = [col for col in df.columns if col not in numeric_cols]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nAttempting grouping by '{group_field}' (first categorical field)")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No categorical/group fields detected for grouping.")
    else:
        print("No numeric fields available for analysis.")
else:
    print("No data frame loaded. Please check data extraction above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Distribution and relationship plot
if dataframes and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field exists, plot average by group
    if group_fields:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Data or numeric fields are not available for plotting.")

## 6. Conclusion
In this notebook, we demonstrated using `mlcroissant` to load FAIR^2 dataset metadata and records, explored field structure and types, conducted filtering/normalization, and visualized key distributions. This process is adaptable to Croissant-compliant datasets thanks to references by record set and field `@id`.

Key insights and next steps could include more advanced groupings, statistical analyses, or integration into automated workflows using the Croissant metadata description.

_Notebook generated for the FAIR^2 dataset and Croissant schema: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json_